In [2]:
# imports

import pandas as pd
import os
import json
import time
from openai import OpenAI
from dotenv import load_dotenv

# load the api key from the .env file in the project folder
load_dotenv('../.env')
client = OpenAI()

model_name = 'gpt-6-luna'

system_prompt = """You rate student reviews of college professors. Read the review and rate it using only what the review says.

leniency: how easy it is to get a good grade. 1 = very harsh grading, 3 = neutral or mixed, 5 = very lenient (easy A, curves, extra credit). null if the review does not talk about grading or how easy the class is.
teaching: quality of the teaching. 1 = very poor (unclear, disorganized, unhelpful), 3 = average or mixed, 5 = excellent (clear, organized, helpful, engaging). null if not mentioned.
workload: amount of work. 1 = very light, 3 = normal, 5 = very heavy. null if not mentioned.
main_reason: the single main thing the review is about. One of "teaching", "grading", "workload", "personality", "other".
grade_complaint: true if the review is mainly complaining about a grade the student received, otherwise false.

Reply with only a JSON object with the keys leniency, teaching, workload, main_reason, grade_complaint. No other text."""

# sends one review to the model and returns its ratings as a dictionary
def score_review(comment):
    response = client.chat.completions.create(
        model=model_name,
        reasoning_effort='none', # keeps cost and time low
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': 'Review: ' + comment}
        ]
    )
    text = response.choices[0].message.content
    return json.loads(text)

In [ ]:
# load the reviews to score
llm_sample = pd.read_csv('../data/samples/llm_sample.csv')
print("Reviews to score:", len(llm_sample))

output_path = '../data/processed/llm_scores.csv'

# skip reviews that were already scored in an earlier run
if os.path.exists(output_path):
    done_ids = set(pd.read_csv(output_path)['review_id'])
else:
    done_ids = set()

print("Already scored:", len(done_ids))

rows = []
for i, row in llm_sample.iterrows():
    if row['review_id'] in done_ids:
        continue
    
    # try up to 3 times if the call or the json fails
    result = None
    for attempt in range(3):
        try:
            result = score_review(row['comment'])
            break
        except Exception as e:
            print("Error on review", row['review_id'], e)
            time.sleep(5)

    if result is None:
        continue # skipped, will be retried on the next run

    result['review_id'] = row['review_id']
    rows.append(result)

    # save every 50 reviews
    if len(rows) == 50:
        pd.DataFrame(rows).to_csv(output_path, mode='a', index=False,
                                  header=not os.path.exists(output_path))
        rows = []
        print("Saved, total scored:", len(pd.read_csv(output_path)))

# save whatever is left at the end
if len(rows) > 0:
    pd.DataFrame(rows).to_csv(output_path, mode='a', index=False,
                              header=not os.path.exists(output_path))

print("Done,", len(pd.read_csv(output_path)), "reviews scored")

Reviews to score: 3000
Keeps the class engaged and teaches based on personal experince rather than out of the textbook, something most professors do not do in today's education system.
{'leniency': None, 'teaching': 5, 'workload': None, 'main_reason': 'teaching', 'grade_complaint': False}

kill yourself before you take dr. hall's A&P I. It is probably not supposed to be THAT hard but he made it so difficult because he cannot teach. He brags about him going to medical school & working in hospitals more than he teaches and he does not know what the hell he is talking about.
{'leniency': None, 'teaching': 1, 'workload': None, 'main_reason': 'teaching', 'grade_complaint': False}

This professor is nice. She would even tutor me when I have questions in homework or quizzes.
{'leniency': None, 'teaching': 5, 'workload': None, 'main_reason': 'teaching', 'grade_complaint': False}

Will definetly do anything to help out students.  Gives out tons of extra credit opportunities!  Has a good sense o